In [3]:
# comprehensive_feature_selection_parallel.py

# Import joblib for parallel processing
from joblib import Parallel, delayed

# ... (all other imports from the original script remain the same)
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_selection import (
    SelectKBest, f_classif, mutual_info_classif, chi2,
    RFE, RFECV, SelectFromModel, VarianceThreshold
)
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LassoCV, LogisticRegression
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import roc_auc_score, classification_report
from imblearn.ensemble import BalancedRandomForestClassifier
from scipy.stats import spearmanr, pearsonr
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import warnings
warnings.filterwarnings('ignore')


# Assume the original ComprehensiveFeatureSelector class is defined here
class ComprehensiveFeatureSelector:
    # ... (the entire original class code)
    def __init__(self, random_state=42):
        self.random_state = random_state
        self.results = {}
        self.scaler = StandardScaler()
        
    def load_and_prepare_data(self, train_path, test_path):
        """Load and prepare the data"""
        print("Loading and preparing data...")
        
        # Load data
        self.train_df = pd.read_pickle(train_path)
        self.test_df = pd.read_pickle(test_path)
        
        # Fill NaNs
        self.train_df.fillna(self.train_df.mean(), inplace=True)
        self.test_df.fillna(self.test_df.mean(), inplace=True)
        
        # Prepare features and targets
        feature_cols = [col for col in self.train_df.columns 
                       if col not in ['graph_id', 'edge_source', 'edge_dest', 'is_causal']]
        
        self.X_train = self.train_df[feature_cols]
        self.y_train = self.train_df['is_causal']
        self.X_test = self.test_df[feature_cols]
        self.y_test = self.test_df['is_causal']
        
        # Scale features
        self.X_train_scaled = pd.DataFrame(
            self.scaler.fit_transform(self.X_train),
            columns=self.X_train.columns,
            index=self.X_train.index
        )
        self.X_test_scaled = pd.DataFrame(
            self.scaler.transform(self.X_test),
            columns=self.X_test.columns,
            index=self.X_test.index
        )
        
        print(f"Training data: {self.X_train.shape}")
        print(f"Test data: {self.X_test.shape}")
        print(f"Class distribution: {self.y_train.value_counts().to_dict()}")
        
    def analyze_feature_characteristics(self):
        """Analyze basic feature characteristics"""
        print("\n" + "="*60)
        print("FEATURE CHARACTERISTICS ANALYSIS")
        print("="*60)
        
        results = {}
        
        # 1. Variance analysis
        variances = self.X_train.var().sort_values(ascending=False)
        low_variance_features = variances[variances < 0.01].index.tolist()
        
        print(f"Features with very low variance (< 0.01): {len(low_variance_features)}")
        if len(low_variance_features) > 0:
            print(f"Examples: {low_variance_features[:5]}")
        
        # 2. Missing value analysis
        missing_counts = self.train_df.isnull().sum()
        features_with_missing = missing_counts[missing_counts > 0]
        print(f"Features that had missing values: {len(features_with_missing)}")
        
        # 3. Feature correlation analysis
        print("\nCalculating feature correlations...")
        corr_matrix = self.X_train.corr().abs()
        
        # Find highly correlated pairs
        high_corr_pairs = []
        for i in range(len(corr_matrix.columns)):
            for j in range(i+1, len(corr_matrix.columns)):
                if corr_matrix.iloc[i, j] > 0.95:
                    high_corr_pairs.append((
                        corr_matrix.columns[i], 
                        corr_matrix.columns[j], 
                        corr_matrix.iloc[i, j]
                    ))
        
        print(f"Feature pairs with correlation > 0.95: {len(high_corr_pairs)}")
        
        # 4. Feature groups analysis
        feature_groups = self._analyze_feature_groups()
        
        results.update({
            'low_variance_features': low_variance_features,
            'high_corr_pairs': high_corr_pairs,
            'feature_groups': feature_groups,
            'correlation_matrix': corr_matrix
        })
        
        self.results['characteristics'] = results
        return results
    
    def _analyze_feature_groups(self):
        """Analyze feature groups by prefix"""
        groups = {}
        for col in self.X_train.columns:
            prefix = col.split('_')[0] if '_' in col else col
            if prefix not in groups:
                groups[prefix] = []
            groups[prefix].append(col)
        
        print(f"\nFeature groups found:")
        for group, features in sorted(groups.items(), key=lambda x: len(x[1]), reverse=True):
            print(f"  {group}: {len(features)} features")
        
        return groups
    
    def remove_low_variance_features(self, threshold=0.01):
        """Remove features with low variance"""
        print(f"\nRemoving features with variance < {threshold}...")
        
        selector = VarianceThreshold(threshold=threshold)
        X_train_filtered = selector.fit_transform(self.X_train)
        X_test_filtered = selector.transform(self.X_test)
        
        selected_features = self.X_train.columns[selector.get_support()].tolist()
        removed_features = self.X_train.columns[~selector.get_support()].tolist()
        
        print(f"Features removed: {len(removed_features)}")
        print(f"Features remaining: {len(selected_features)}")
        
        return {
            'method': 'variance_threshold',
            'selected_features': selected_features,
            'removed_features': removed_features,
            'X_train': X_train_filtered,
            'X_test': X_test_filtered
        }
    
    def correlation_based_selection(self, threshold=0.95):
        """Remove highly correlated features"""
        print(f"\nRemoving features with correlation > {threshold}...")
        
        corr_matrix = self.X_train.corr().abs()
        
        # Find features to remove
        upper_tri = corr_matrix.where(
            np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
        )
        
        to_drop = [column for column in upper_tri.columns 
                  if any(upper_tri[column] > threshold)]
        
        selected_features = [col for col in self.X_train.columns if col not in to_drop]
        
        print(f"Features removed due to high correlation: {len(to_drop)}")
        print(f"Features remaining: {len(selected_features)}")
        
        return {
            'method': 'correlation_threshold',
            'selected_features': selected_features,
            'removed_features': to_drop,
            'threshold': threshold
        }
    
    def _evaluate_selection(self, X_train, X_test, n_jobs=1):
        """Evaluate feature selection using BalancedRandomForest"""
        # n_jobs is passed to control inner parallelism
        clf = BalancedRandomForestClassifier(
            n_estimators=100,
            random_state=self.random_state,
            n_jobs=n_jobs
        )
        
        clf.fit(X_train, self.y_train)
        y_pred_proba = clf.predict_proba(X_test)[:, 1]
        auc = roc_auc_score(self.y_test, y_pred_proba)
        
        return auc
    # ... The rest of the original methods would be here ...


class ParallelComprehensiveFeatureSelector(ComprehensiveFeatureSelector):
    def __init__(self, random_state=42, n_jobs=-1):
        """
        Initialize the parallel selector.
        n_jobs: Number of cores to use for parallel execution. -1 means all available.
        """
        super().__init__(random_state)
        self.n_jobs = n_jobs if n_jobs != -1 else os.cpu_count()
        print(f"Parallel selector initialized to use {self.n_jobs} cores.")

    # Override _evaluate_selection to manage n_jobs
    def _evaluate_selection(self, X_train, X_test, n_jobs=1):
        """
        Internal evaluation function.
        By default, it uses n_jobs=1 to avoid nested parallelism when called from a parallel loop.
        """
        clf = BalancedRandomForestClassifier(
            n_estimators=100,
            random_state=self.random_state,
            n_jobs=1
        )
        clf.fit(X_train, self.y_train)
        y_pred_proba = clf.predict_proba(X_test)[:, 1]
        return roc_auc_score(self.y_test, y_pred_proba)

    # --- Worker functions for parallel execution ---
    
    def _run_univariate_worker(self, params):
        score_name, score_func, k = params
        if k > self.X_train.shape[1]:
            return None
            
        selector = SelectKBest(score_func=score_func, k=k)
        X_train_selected = selector.fit_transform(self.X_train_scaled, self.y_train)
        X_test_selected = selector.transform(self.X_test_scaled)
        
        selected_features = self.X_train.columns[selector.get_support()].tolist()
        auc_score = self._evaluate_selection(X_train_selected, X_test_selected, n_jobs=1) # n_jobs=1
        
        print(f"  Finished Univariate: {score_name}, k={k}, AUC={auc_score:.4f}")
        return score_name, k, {'selected_features': selected_features, 'scores': selector.scores_, 'auc': auc_score}

    def _run_rfe_worker(self, step):
        estimator = BalancedRandomForestClassifier(
            n_estimators=100, random_state=self.random_state, n_jobs=1 # n_jobs=1
        )
        selector = RFECV(
            estimator=estimator, step=step,
            cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=self.random_state),
            scoring='roc_auc', n_jobs=self.n_jobs // len(self.step_sizes_for_rfe) # Distribute cores
        )
        X_train_selected = selector.fit_transform(self.X_train_scaled, self.y_train)
        X_test_selected = selector.transform(self.X_test_scaled)
        selected_features = self.X_train.columns[selector.get_support()].tolist()
        
        # Final evaluation on full test set
        auc_score = self._evaluate_selection(X_train_selected, X_test_selected, n_jobs=1) # n_jobs=1
        
        print(f"  Finished RFE: step={step}, Features={selector.n_features_}, AUC={auc_score:.4f}")
        return f'step_{step}', {'selected_features': selected_features, 'n_features': selector.n_features_, 'cv_scores': selector.cv_results_, 'auc': auc_score}

    def _run_model_based_worker(self, model_params):
        model_name, model = model_params
        
        if model_name == 'lasso':
            model.fit(self.X_train_scaled, self.y_train)
            selector = SelectFromModel(model, prefit=True)
        else:
            selector = SelectFromModel(model)
            selector.fit(self.X_train_scaled, self.y_train)
            
        X_train_selected = selector.transform(self.X_train_scaled)
        X_test_selected = selector.transform(self.X_test_scaled)
        selected_features = self.X_train.columns[selector.get_support()].tolist()
        
        auc_score = self._evaluate_selection(X_train_selected, X_test_selected, n_jobs=1) # n_jobs=1
        
        print(f"  Finished Model-based: {model_name}, Features={len(selected_features)}, AUC={auc_score:.4f}")
        return model_name, {'selected_features': selected_features, 'n_features': len(selected_features), 'auc': auc_score}

    # --- Overridden methods to use parallel execution ---

    def univariate_selection(self, k_values=[10, 25, 50, 100, 200]):
        print("\n" + "="*60)
        print("UNIVARIATE FEATURE SELECTION (PARALLEL)")
        print("="*60)
        
        scoring_functions = {'f_classif': f_classif, 'mutual_info': mutual_info_classif}
        tasks = [(name, func, k) for name, func in scoring_functions.items() for k in k_values]
        
        with Parallel(n_jobs=self.n_jobs) as parallel:
            results_list = parallel(delayed(self._run_univariate_worker)(task) for task in tasks)

        # Process results
        results = {name: {} for name in scoring_functions.keys()}
        for result in results_list:
            if result:
                score_name, k, data = result
                results[score_name][k] = data
        
        self.results['univariate'] = results
        return results

    def recursive_feature_elimination(self, step_sizes=[1, 5]):
        print("\n" + "="*60)
        print("RECURSIVE FEATURE ELIMINATION (PARALLEL)")
        print("="*60)
        
        self.step_sizes_for_rfe = step_sizes # Store for worker
        
        # Here we parallelize the outer loop over step_sizes.
        # RFECV itself is parallel, so we allocate a subset of cores to each RFECV instance.
        with Parallel(n_jobs=len(step_sizes)) as parallel:
            results_list = parallel(delayed(self._run_rfe_worker)(step) for step in step_sizes)
        
        self.results['rfe'] = dict(results_list)
        return self.results['rfe']

    def model_based_selection(self):
        print("\n" + "="*60)
        print("MODEL-BASED FEATURE SELECTION (PARALLEL)")
        print("="*60)

        # Configure models to use a fraction of cores, as they run in parallel
        # We give each model a chunk of the available cores for their internal CV/fitting
        num_models = 3
        inner_jobs = 2
        
        models = {
            'random_forest': BalancedRandomForestClassifier(n_estimators=100, random_state=self.random_state, n_jobs=inner_jobs),
            'lasso': LassoCV(cv=5, random_state=self.random_state, max_iter=1000, n_jobs=inner_jobs),
            'logistic_l1': LogisticRegression(penalty='l1', solver='liblinear', random_state=self.random_state, C=1.0) # liblinear is serial
        }
        
        with Parallel(n_jobs=len(models)) as parallel:
            results_list = parallel(delayed(self._run_model_based_worker)(item) for item in models.items())
            
        self.results['model_based'] = dict(results_list)
        return self.results['model_based']

    def run_full_pipeline(self, train_path, test_path):
        """Run the complete feature selection pipeline IN PARALLEL."""
        print("STARTING PARALLEL COMPREHENSIVE FEATURE SELECTION PIPELINE")
        print("="*80)
        
        self.load_and_prepare_data(train_path, test_path)
        
        # These are fast and can run sequentially
        self.analyze_feature_characteristics()
        
        # Run major analyses in parallel where possible
        self.univariate_selection()
        self.recursive_feature_elimination()
        self.model_based_selection()
        
        # These are either fast or not easily parallelized at a high level
        self.clustering_based_selection()
        self.pca_based_analysis()
        
        # Compare and visualize
        self.compare_methods()
        self.create_visualizations()
        
        best_sets = self.get_best_feature_sets()
        
        print("\n" + "="*80)
        print("PIPELINE COMPLETED!")
        print("="*80)
        
        return self.results, best_sets

In [13]:
import os
import pickle


selector = ParallelComprehensiveFeatureSelector(random_state=42, n_jobs=60)
train_path = 'data/descriptors_df_train_newdynamic.pkl'
test_path = 'data/descriptors_netsim_5_full.pkl'
 
selector.load_and_prepare_data(train_path, test_path)
        
       

Parallel selector initialized to use 60 cores.
Loading and preparing data...


KeyError: "['forward_transfer_cmi_lag_1', 'backward_transfer_cmi_lag_1', 'forward_transfer_cmi_lag_2', 'backward_transfer_cmi_lag_2', 'forward_transfer_cmi_lag_3', 'backward_transfer_cmi_lag_3'] not in index"

In [9]:
# These are fast and can run sequentially
selector.analyze_feature_characteristics()
        
     


FEATURE CHARACTERISTICS ANALYSIS
Features with very low variance (< 0.01): 23
Examples: ['copula_tau_4', 'errors_correlation_with_inputs', 'backward_transfer_cmi_q6', 'forward_transfer_cmi_q6', 'backward_transfer_cmi_q5']
Features that had missing values: 0

Calculating feature correlations...
Feature pairs with correlation > 0.95: 225

Feature groups found:
  mca: 21 features
  cau: 16 features
  eff: 16 features
  m: 14 features
  forward: 9 features
  backward: 9 features
  copula: 7 features
  mbe: 7 features
  HOC: 4 features
  n: 3 features
  coeff: 2 features
  kurtosis: 2 features
  skewness: 2 features
  parcorr: 1 features
  errors: 1 features
  com: 1 features


{'low_variance_features': ['copula_tau_4',
  'errors_correlation_with_inputs',
  'backward_transfer_cmi_q6',
  'forward_transfer_cmi_q6',
  'backward_transfer_cmi_q5',
  'forward_transfer_cmi_q5',
  'backward_transfer_cmi_q4',
  'forward_transfer_cmi_q4',
  'backward_transfer_cmi_q3',
  'backward_transfer_cmi_mean',
  'forward_transfer_cmi_q3',
  'forward_transfer_cmi_mean',
  'backward_transfer_cmi_q2',
  'backward_transfer_cmi_std',
  'forward_transfer_cmi_q2',
  'forward_transfer_cmi_std',
  'backward_transfer_cmi_q1',
  'forward_transfer_cmi_q1',
  'backward_transfer_cmi_q0',
  'forward_transfer_cmi_q0',
  'n_features/n_samples',
  'n_samples',
  'n_features'],
 'high_corr_pairs': [('forward_transfer_cmi_q0',
   'forward_transfer_cmi_q1',
   0.9938607597688279),
  ('forward_transfer_cmi_q1', 'forward_transfer_cmi_q2', 0.9723201286445273),
  ('forward_transfer_cmi_q2', 'forward_transfer_cmi_q3', 0.9786129865064355),
  ('forward_transfer_cmi_q4', 'forward_transfer_cmi_q5', 0.98612336

In [11]:
selector.recursive_feature_elimination()


RECURSIVE FEATURE ELIMINATION (PARALLEL)


KeyboardInterrupt: 

In [ ]:
   # Run major analyses in parallel where possible
       
        self.
        self.model_based_selection()
        
        # These are either fast or not easily parallelized at a high level
        self.clustering_based_selection()
        self.pca_based_analysis()
        
        # Compare and visualize
        self.compare_methods()
        self.create_visualizations()
        
        best_sets = self.get_best_feature_sets()


# Run the full parallel pipeline
results, best_feature_sets = selector.run_full_pipeline(
    'data/descriptors_df_train.pkl',
    'data/descriptors_netsim_5_full.pkl'
)

# Save results
with open('feature_selection_results_parallel.pkl', 'wb') as f:
    pickle.dump({
        'results': results,
        'best_feature_sets': best_feature_sets
    }, f)

print("\nParallel results saved to 'feature_selection_results_parallel.pkl'")